# 05 - Feature Engineering Experiment

            Mục tiêu:

            - Kiểm tra output giai đoạn feature engineering.
            - Kiểm tra feature metadata.
            - Kiểm tra train/validation/test split.
            - Kiểm tra missing/correlation để đảm bảo data sẵn sàng train model.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: e:\fraud-detection-project


In [2]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_path = PROCESSED_DIR / "model_train_features.parquet"
val_path = PROCESSED_DIR / "model_validation_features.parquet"
test_path = PROCESSED_DIR / "model_test_features.parquet"
metadata_path = PROCESSED_DIR / "feature_metadata.json"

for path in [train_path, val_path, test_path, metadata_path]:
    print(path, path.exists())

e:\fraud-detection-project\data\processed\model_train_features.parquet True
e:\fraud-detection-project\data\processed\model_validation_features.parquet True
e:\fraud-detection-project\data\processed\model_test_features.parquet True
e:\fraud-detection-project\data\processed\feature_metadata.json True


In [3]:
train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head(3).T.rename(columns=lambda i: f"row_{i}"))

Train: (70000, 59)
Validation: (15000, 59)
Test: (15000, 59)


,row_0,row_1,row_2
night_ratio_30d,0.76,0.236,0.099
mcc_entropy_30d,2.124,1.742,2.48
distinct_merchants_7d,21,120,1
distinct_countries_30d,10,20,5
device_diversity_30d,32,24,3
decline_rate_30d,0.282,0.095,0.101
chargebacks_365d,0,0,0
credit_util_today,1.452,0.328,1.486
spending_trend,4.548,4.655,3.255
ip_score,0.3064,0.4186,0.171


In [4]:
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

for key in ["numeric_features", "categorical_features", "binary_features", "interaction_features"]:
    print("\n" + "=" * 100)
    values = metadata.get(key, [])
    print(key, len(values))
    print(values[:15])
    if len(values) > 15:
        print(f"... {len(values) - 15} more")


numeric_features 19
['night_ratio_30d', 'mcc_entropy_30d', 'distinct_merchants_7d', 'distinct_countries_30d', 'device_diversity_30d', 'decline_rate_30d', 'chargebacks_365d', 'credit_util_today', 'spending_trend', 'ip_score', 'txn_count_7d', 'txn_count_30d', 'txn_count_ratio_7d_30d', 'hour', 'log_mean_amount_30d']
... 4 more

categorical_features 21
['currency', 'payment_channel', 'merchant_country', 'mcc', 'card_entry_mode', 'auth_result', 'pin_verif_method', 'recurring_flag', 'auth_characteristics', 'message_type', 'term_location', 'day_name', 'time_period', 'night_ratio_group', 'night_unusual_group']
... 6 more

binary_features 10
['very_low_mcc_entropy_flag', 'low_mcc_entropy_flag', 'low_night_ratio_flag', 'high_max_amount_30d_flag', 'low_spending_trend_flag', 'low_country_diversity_flag', 'night_transaction_flag', 'card_not_present_flag', 'cross_border_flag', 'tokenised_flag']

interaction_features 8
['low_mcc_entropy_x_low_night_ratio', 'low_mcc_entropy_x_low_night_ratio_x_card_n

In [5]:
for name, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print("\n" + "=" * 100)
    print(name)
    target_dist = d["fraud"].value_counts(normalize=True).rename("ratio").reset_index()
    display(target_dist)


train


,fraud,ratio
0,0,0.974529
1,1,0.025471



validation


,fraud,ratio
0,0,0.9748
1,1,0.0252



test


,fraud,ratio
0,0,0.974133
1,1,0.025867


In [6]:
all_features = metadata["all_model_features"]

missing = (
    train_df[all_features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing.columns = ["feature", "missing_rate"]
display(missing.head(40))

,feature,missing_rate
0,night_ratio_30d,0.0
1,mcc_entropy_30d,0.0
2,distinct_merchants_7d,0.0
3,distinct_countries_30d,0.0
4,device_diversity_30d,0.0
5,decline_rate_30d,0.0
6,chargebacks_365d,0.0
7,credit_util_today,0.0
8,spending_trend,0.0
9,ip_score,0.0


In [7]:
numeric_features = metadata["numeric_features"]
corr = train_df[numeric_features].corr(numeric_only=True)

corr_abs_pairs = (
    corr.abs()
    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
    .reset_index()
)
corr_abs_pairs.columns = ["feature_1", "feature_2", "abs_corr"]

display(corr_abs_pairs.head(30))

,feature_1,feature_2,abs_corr
0,amount_to_mean_30d,amount_z_30d,0.764547
1,txn_count_30d,txn_count_ratio_7d_30d,0.593468
2,amount_to_mean_30d,log_amount_to_max_30d,0.389256
3,txn_count_7d,txn_count_ratio_7d_30d,0.383177
4,log_amount_to_max_30d,amount_z_30d,0.366764
5,log_mean_amount_30d,amount_to_mean_30d,0.362353
6,night_ratio_30d,night_unusual_score,0.338724
7,hour,night_unusual_score,0.327516
8,txn_count_7d,txn_count_30d,0.251589
9,night_ratio_30d,amount_z_30d,0.013876


In [8]:
print("Dropped columns:", metadata.get("dropped_columns"))
print("Thresholds:", metadata.get("thresholds"))

Dropped columns: ['ip_risk', 'txn_counts', 'ip_address', 'transaction_datetime']
Thresholds: {'very_low_mcc_entropy_q05': 0.197, 'low_mcc_entropy_q10': 0.393, 'low_night_ratio_q10': 0.101, 'high_max_amount_q90': 225176.777, 'low_spending_trend_q10': 0.584, 'low_country_diversity_q10': 3.0, 'high_device_diversity_q90': 36.0, 'high_decline_rate_q90': 0.283, 'high_ip_score_q90': 0.5123}


## Kết luận cần ghi sau khi chạy

            - Feature set đã đủ cho model chưa?
            - Có feature missing bất thường không?
            - Có feature tương quan cao cần loại cho Logistic Regression không?
            - Với CatBoost, interaction thủ công có thể bỏ được không?